# Kombinierte Labels: PCA auf Spektralbändern

**Hypothese:** Stark korrelierte Spektralbänder erschweren die Abstandsmessung des RBF-Kernels. Eine fold-gefittete PCA könnte die relevanten Unterschiede in weniger, unkorrelierten Komponenten bündeln.

**Versuchsplan:** Es werden genau 20, 40 und 80 PCA-Komponenten verglichen. AEZ und Month umgehen die PCA. Die Auswahl von Komponentenanzahl sowie `C` und `gamma` erfolgt ausschließlich über die fünf festen Trainingsfolds; nur der Gesamtsieger wird einmal auf dem gemeinsamen Validierungssplit bewertet.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig

PCA_COMPONENT_COUNTS = (20, 40, 80)
SVM_PARAM_GRID = {
    "estimator__C": [100.0, 300.0],
    "estimator__gamma": [0.001, 0.003],
}
REFERENCE_CV_BACC_COMBINED = 0.8473
REFERENCE_VALIDATION_BACC_COMBINED = 0.8664

## 1. PCA- und SVM-Kandidaten mit Cross-Validation vergleichen

Jede PCA wird innerhalb ihrer jeweiligen Pipeline in jedem Fold neu gefittet. Die SVM-Suche bleibt klein und umfasst die Referenzumgebung sowie den besten CV-only-Kandidaten der vorigen Sensitivitätsanalyse.

In [ ]:
tuned_by_components = {}
for component_count in PCA_COMPONENT_COUNTS:
    preprocessing = PreprocessingConfig(
        scale=True,
        use_meta=True,
        pca_components=component_count,
    )
    tuned_by_components[component_count] = tune(
        make_combined_svm(),
        TuneConfig(
            name=f"combined_svm_pca_{component_count}_search",
            approach="combined",
            description=(
                f"RBF-SVM auf {component_count} fold-gefitten PCA-Komponenten der Spektralbänder "
                "mit AEZ und Month außerhalb der PCA."
            ),
            preprocessing=preprocessing,
            param_grid=SVM_PARAM_GRID,
        ),
    )

candidate_summary = (
    pd.DataFrame(
        {
            "PCA-Komponenten": component_count,
            "CV BAcc kombiniert": tuned.best_score,
            "Beste Parameter": tuned.best_params,
        }
        for component_count, tuned in tuned_by_components.items()
    )
    .sort_values("CV BAcc kombiniert", ascending=False)
)
candidate_summary

## 2. Den CV-Gewinner einmal validieren

Die Auswahl berücksichtigt ausschließlich die CV-Rangfolge. Die Validierung beantwortet danach nur noch, ob die ausgewählte PCA-Variante die Rohspektren-Referenz auf dem gemeinsamen Holdout übertrifft.

In [ ]:
best_component_count = int(candidate_summary.iloc[0]["PCA-Komponenten"])
best_tuned = tuned_by_components[best_component_count]
best_preprocessing = PreprocessingConfig(
    scale=True,
    use_meta=True,
    pca_components=best_component_count,
)

result = run(
    best_tuned.best_model,
    RunConfig(
        name="combined_svm_pca",
        approach="combined",
        description=(
            "Beste RBF-SVM mit PCA nur auf skalierten Spektralbändern, anhand der "
            "Trainings-CV ausgewählt und einmal auf dem gemeinsamen Validierungssplit bewertet."
        ),
        preprocessing=best_preprocessing,
        tuning_run=best_tuned.run_id,
    ),
)

{
    "PCA-Komponenten": best_component_count,
    "CV BAcc kombiniert": best_tuned.best_score,
    "CV-Parameter": best_tuned.best_params,
    "Validierungsmetriken": result.metrics.model_dump(),
    "Tuning-Run": best_tuned.run_id,
    "Bewertungs-Run": result.run_id,
}

## 3. Ergebnis prüfen

Die Analyse stellt die gewählte PCA-Variante der aktuellen Rohspektren-Referenz gegenüber. Alle Vorhersagen müssen weiterhin gültige Crop/Stage-Paare sein.

In [ ]:
from IPython.display import display

from awp2.evaluation import plot_confusion_matrices
from awp2.plots import (
    plot_class_recall,
    plot_metric_comparison,
    plot_top_confusions,
    plot_tuning_candidates,
)

reference_metrics = result.metrics.model_copy(
    update={
        "bacc_crop": 0.931,
        "bacc_stage": 0.906,
        "bacc_combined": REFERENCE_VALIDATION_BACC_COMBINED,
    }
)
metric_ax = plot_metric_comparison(
    {
        "Rohspektren-Referenz": reference_metrics,
        f"PCA ({best_component_count})": result.metrics,
    },
)
metric_ax.set_title("Kombinierte Labels: PCA gegen Rohspektren")
display(metric_ax.figure)

tuning_ax = plot_tuning_candidates(
    best_tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title(f"RBF-SVM mit {best_component_count} PCA-Komponenten")
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

assert result.metrics.invalid_combinations == 0.0

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

save_doc_figure(metric_ax.figure, "combined_svm_pca_comparison", MODEL_DOCS_FIGURES_DIR)
save_doc_figure(tuning_ax.figure, "combined_svm_pca_tuning", MODEL_DOCS_FIGURES_DIR)
save_doc_figure(
    plot_confusion_matrices(result.y_val, result.y_pred),
    "combined_svm_pca_confusion",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_class_recall(result.y_val, result.y_pred),
    "combined_svm_pca_recall",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_top_confusions(result.y_val, result.y_pred),
    "combined_svm_pca_confusions",
    MODEL_DOCS_FIGURES_DIR,
)

## Fazit

Die PCA-Variante wird nur weiterverfolgt, wenn ihre kombinierte Balanced Accuracy auf dem einmalig verwendeten Validierungssplit die Rohspektren-Referenz von `0.8664` übertrifft. Andernfalls bleibt `combined_svm_expanded` die Referenz; das Ergebnis wird dennoch mit CV- und Validierungswerten dokumentiert.